# Final Project, Part 2: An LSTM That Gives a Range (Rama llama's version)
**OPIM 5509 · leaderboard nickname: Rama llama · NetID raa20002**

Dr. Dave's FP2 notebook, Parts 0-9, with my **"Your turn"** changes built in. The live model gives three numbers for
every hour of the next 24: a **low** (10th percentile), an **expected** value (50th) and a **high** (90th), an 80% range.

**What I changed from the FP2 starter (and why)**

| Change | Why |
| :-- | :-- |
| **GRU encoder-decoder** instead of one LSTM over a 192-row table: `Conv1D` + `GRU` read the past week, a second `GRU` walks the next 24 hours | each forecast hour gets its own row: its weather, its calendar, its load anchors |
| **5 networks averaged** inside one `model.keras` (an `Average` layer) | averaging 5 random starts cut the CRPS about 5 MW on 2025 |
| **Load anchors** in every future row: same hour yesterday, same hour last week, the newest finished hour | on calm days, "same hour yesterday" is hard to beat: the network can copy yesterday's shape and adjust it |
| **Sunlight** (`solar_wm2`) as a 6th weather column | rooftop solar hides load at midday (worth ~3 MW of CRPS on 2025) |
| **Trained on HRRR weather forecasts as issued** (the run that really existed ~7 hours after it started), not the weather that happened | live, the model only ever sees a forecast, so it learns how far to trust one; the cloud gives it the newest HRRR run from the class file (NWS forecast as the fallback) |
| **Final fit on every hour through Oct 2026** | load drifts (more solar, more electric heat): a model trained through 2025 scored 56.6 vs 64.6 MW CRPS on Aug-Oct 2026 for one trained through 2024 |

**How the numbers in this notebook are made.** Three ensembles, all from `train_ensemble` in Part 5:
`model_2024` (trained 2023-2024: the 2025 validation and calibration), `model_2025` (trained 2023-2025: the one look at
2026) and `model_final` (trained through Oct 2026: the live model). Training 3 x 5 networks takes ~25 minutes on a T4,
so by default this notebook **loads the three ensembles exactly as I trained them** (saved in my repo) and recomputes
every score from them. Set `RETRAIN = True` to train all three from scratch here instead (scores land within ~1 MW;
GPUs and random starts differ a little).

**Runtime > Change runtime type > T4 GPU.**

## Part 0 · Before you start

In [ ]:
import io, json, os, time, urllib.request
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from keras.callbacks import EarlyStopping
from pandas.tseries.holiday import USFederalHolidayCalendar

RETRAIN = False          # False: load my three trained ensembles from my repo (exact numbers). True: train them here.
REPO = "https://raw.githubusercontent.com/RamaAkach/energy-forecasting-dl/main/"

SETTINGS = {
    "seed": 5509,
    "lookback_hours": 168,       # the past week of load
    "horizon_hours": 24,         # the next 24 hours
    "hrrr_lag_hours": 7,         # an HRRR run reaches the class file ~7 hours after it starts
    "units": 128,                # GRU size
    "conv_filters": 32,
    "members": 5,                # networks averaged in the ensemble
    "epochs": 12,                # fixed: early stopping on 2025 picked 9-19 epochs (Part 5's loss curve)
    "batch_size": 256,
}
QUANTILES = [0.1, 0.5, 0.9]      # low, expected, high
keras.utils.set_random_seed(SETTINGS["seed"])
print("Keras", keras.__version__, "| GPU:", len(tf.config.list_physical_devices("GPU")) > 0, "| RETRAIN:", RETRAIN)

## Part 1 · Load the data
The same hourly file as FP1/FP2 (Connecticut load and Bradley weather, 2023 to Oct 2026), plus every **HRRR forecast
for Bradley as it was issued** (4 runs a day, 48 hours ahead, with sunlight) from the course's OPIM 5512 data folder.

In [ ]:
BASE = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(BASE + "bdl_ct_load_hourly_2023_2026.csv", parse_dates=["hour_utc"]).set_index("hour_utc").sort_index()
df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="h"))
hrrr = pd.read_csv(BASE + "bdl_hrrr_asissued_2023_2026.csv.gz", parse_dates=["init_utc", "hour_utc"])
WEATHER_COLUMNS = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct", "solar_wm2"]
print("load:", df.shape, "|", df.index.min(), "to", df.index.max())
print("HRRR runs as issued:", hrrr["init_utc"].nunique(), "|", hrrr["init_utc"].min(), "to", hrrr["init_utc"].max())

## Part 2 · CRPS: how the leaderboard scores a range
**CRPS = ( 0.5 × |actual − expected| + 0.1 × width + 1 × outside ) ÷ 1.5**, in MW, lower is better. It equals the three
pinball losses (q = 0.1, 0.5, 0.9) added up ÷ 1.5, so training on the pinball loss trains on the leaderboard's score.
**CRPS skill** = 1 − my CRPS ÷ same hour yesterday's error, in each lead block (1-6, 7-12, 13-18, 19-24 h), averaged.

In [ ]:
def crps(actual, low, expected, high):
    width = high - low
    outside = np.clip(low - actual, 0, None) + np.clip(actual - high, 0, None)
    return (0.5 * np.abs(actual - expected) + 0.1 * width + outside) / 1.5


def pinball(actual, forecast, q):
    gap = np.asarray(actual, dtype=float) - np.asarray(forecast, dtype=float)
    return np.maximum(q * gap, (q - 1) * gap)


def crps_skill(scores, baseline_error):
    skills = [1 - np.nanmean(scores[:, a:b]) / np.nanmean(baseline_error[:, a:b]) for a, b in [(0, 6), (6, 12), (12, 18), (18, 24)]]
    return float(np.mean(skills))


# the CRPS is the pinball losses / 1.5: check on Dr. Dave's four forecasters
low, mid, high, actual = np.array([3000, 2980, 2850, 2000]), 3000, np.array([3000, 3020, 3200, 4000]), 3100
print("CRPS:            ", crps(actual, low, mid, high).round(1))
print("pinball sum / 1.5:", ((pinball(actual, low, .1) + pinball(actual, mid, .5) + pinball(actual, high, .9)) / 1.5).round(1))

## Part 3 · The inputs, built exactly like the cloud
**`make_inputs` is the contract with `forecast-24h-dl`** (my fork has a word-for-word copy, `make_inputs_rama_v1`,
picked by `"feature_set": "rama_v1"` in the model card). Two inputs for one forecast issued at hour T:

* **past** `(168, 5)`: the scaled load for T-168 ... T-1, hour of day and day of week (sine, cosine)
* **future** `(24, 17)`: for T+1 ... T+24, the 6 HRRR weather columns (scaled), the 8 calendar columns, and 3 **load
  anchors**: the same hour yesterday (2 days back for T+24, whose "yesterday" is hour T, not over yet), the same hour
  last week, and the newest finished hour T-1.

For ~66,000 training samples, `build_samples` builds the same numbers for every hour at once; the check proves both
ways give identical inputs. Each issue hour uses the **newest HRRR run that existed** (started 7+ hours before T);
the run before it is extra training data.

In [ ]:
HOLIDAYS = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
LB, H = SETTINGS["lookback_hours"], SETTINGS["horizon_hours"]


def calendar_features(hours_utc):
    """Hour of day, day of week and time of year as sines and cosines, plus weekend and holiday flags, in Connecticut time."""
    local = pd.DatetimeIndex(hours_utc).tz_convert("America/New_York")
    out = pd.DataFrame(index=pd.DatetimeIndex(hours_utc))
    out["hour_sin"] = np.sin(2 * np.pi * local.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * local.hour / 24)
    out["dow_sin"] = np.sin(2 * np.pi * local.dayofweek / 7)
    out["dow_cos"] = np.cos(2 * np.pi * local.dayofweek / 7)
    out["doy_sin"] = np.sin(2 * np.pi * local.dayofyear / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * local.dayofyear / 365.25)
    out["is_weekend"] = (local.dayofweek >= 5).astype(float)
    out["is_holiday"] = local.tz_localize(None).normalize().isin(HOLIDAYS).astype(float)
    return out


def make_inputs(issue_hour_utc, load, weather, card):
    """THE CONTRACT (forecast-24h-dl has the same code as make_inputs_rama_v1). Two inputs for ONE forecast."""
    issue = pd.Timestamp(issue_hour_utc)
    lookback = int(card["lookback_hours"])
    horizon = int(card["horizon_hours"])
    past_hours = pd.date_range(issue - pd.Timedelta(hours=lookback), periods=lookback, freq="h")
    future_hours = pd.date_range(issue + pd.Timedelta(hours=1), periods=horizon, freq="h")
    past_load = (load.reindex(past_hours).values - card["load_mean"]) / card["load_std"]
    past_cal = calendar_features(past_hours)[["hour_sin", "hour_cos", "dow_sin", "dow_cos"]].values
    past = np.column_stack([past_load, past_cal])
    columns = []
    for column in card["weather_columns"]:
        values = weather[column].reindex(future_hours).values
        columns.append((values - card["weather_mean"][column]) / card["weather_std"][column])
    future = np.column_stack(columns + [calendar_features(future_hours).values])
    # position j in past_load is hour T-168+j: the same hour yesterday of T+h is j = 144+h (h = 1..23), T+24 -> j = 144
    yesterday = past_load[np.r_[144 + np.arange(1, 24), 144]]
    last_week = past_load[np.arange(1, 25)]
    newest = np.repeat(past_load[-1], 24)
    future = np.column_stack([future, yesterday, last_week, newest])
    return past.astype("float32"), future.astype("float32")


PAST_COLUMNS = ["load_mw", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
FUTURE_COLUMNS = WEATHER_COLUMNS + list(calendar_features(df.index[:1]).columns) + ["load_same_hour_yesterday", "load_same_hour_last_week", "load_newest_hour"]

In [ ]:
def build_samples(run_offset=0):
    """Every issue hour T at once: the past week of load (MW), the HRRR run that was newest at T (run_offset=1: the
    one before), the calendar, and the answer y (the next 24 hours of load, MW)."""
    hours = df.index
    load = df["load_mw"].values.astype("float64")
    inits = np.array(sorted(hrrr["init_utc"].dt.tz_convert(None).unique())).astype("datetime64[ns]")
    index_of = {pd.Timestamp(t).tz_localize("UTC"): i for i, t in enumerate(inits)}
    W = np.full((len(inits), 49, len(WEATHER_COLUMNS)), np.nan)          # run x lead x weather column
    W[hrrr["init_utc"].map(index_of).values, hrrr["lead_hours"].values] = hrrr[WEATHER_COLUMNS].values
    cal = calendar_features(hours).values.astype("float32")
    T_idx = np.arange(LB, len(hours) - H)
    T = hours[T_idx]
    available = (T - pd.Timedelta(hours=SETTINGS["hrrr_lag_hours"])).tz_convert(None).values.astype("datetime64[ns]")
    k = np.searchsorted(inits, available, side="right") - 1 - run_offset
    keep = k >= 0
    T_idx, T, k = T_idx[keep], T[keep], k[keep]
    lead0 = ((T.tz_convert(None).values.astype("datetime64[ns]") - inits[k]) / np.timedelta64(1, "h")).astype(int) + 1
    leads = lead0[:, None] + np.arange(H)[None, :]
    keep = leads[:, -1] <= 48
    T_idx, T, k, leads = T_idx[keep], T[keep], k[keep], leads[keep]
    fw = W[k[:, None], leads]
    past = np.stack([load[T_idx - LB + j] for j in range(LB)], 1)
    fut_i = T_idx[:, None] + 1 + np.arange(H)[None, :]
    y = load[fut_i]
    good = ~(np.isnan(past).any(1) | np.isnan(y).any(1) | np.isnan(fw).any((1, 2)))
    return {"T": T[good], "past": past[good], "y": y[good], "fw": fw[good], "init": inits[k][good],
            "past_cal": cal[T_idx[:, None] - LB + np.arange(LB)[None, :]][good], "fut_cal": cal[fut_i][good]}


def make_card(train_end):
    """The scaling numbers, from the training hours only."""
    train_hours = df.index < train_end
    first = newest_run["fw"][(newest_run["T"] + pd.Timedelta(hours=24)) < train_end].reshape(-1, len(WEATHER_COLUMNS))
    return {"model_type": "keras_gru_ensemble", "feature_set": "rama_v1", "lookback_hours": LB, "horizon_hours": H,
            "weather_columns": WEATHER_COLUMNS,
            "load_mean": float(df.loc[train_hours, "load_mw"].mean()), "load_std": float(df.loc[train_hours, "load_mw"].std()),
            "weather_mean": {c: float(first[:, i].mean()) for i, c in enumerate(WEATHER_COLUMNS)},
            "weather_std": {c: float(first[:, i].std()) for i, c in enumerate(WEATHER_COLUMNS)}}


def features(d, card):
    """make_inputs for every sample at once (fast)."""
    p = (d["past"] - card["load_mean"]) / card["load_std"]
    wm = np.array([card["weather_mean"][c] for c in WEATHER_COLUMNS]); ws = np.array([card["weather_std"][c] for c in WEATHER_COLUMNS])
    past = np.concatenate([p[:, :, None], d["past_cal"][:, :, :4]], -1)
    j24 = np.r_[144 + np.arange(1, 24), 144]
    fut = np.concatenate([(d["fw"] - wm) / ws, d["fut_cal"], p[:, j24][:, :, None], p[:, 1:25][:, :, None],
                          np.repeat(p[:, -1:, None], 24, 1)], -1)
    return past.astype("float32"), fut.astype("float32"), ((d["y"] - card["load_mean"]) / card["load_std"]).astype("float32")


def subset(d, first_issue, last_target):
    m = (d["T"] >= first_issue) & ((d["T"] + pd.Timedelta(hours=24)) < last_target)
    return {k: v[m] for k, v in d.items()}


U = lambda s: pd.Timestamp(s, tz="UTC")
start = time.time()
newest_run = build_samples(0)
previous_run = build_samples(1)
print(f"built in {time.time() - start:.0f} s:", len(newest_run["T"]), "issue hours (+", len(previous_run["T"]), "with the run before)")

# ---- The check: make_inputs (the cloud's code) == features (the training code), on 8 random samples ----
check_card = make_card(U("2025-01-01"))
P, F, _ = features(newest_run, check_card)
rng = np.random.default_rng(SETTINGS["seed"])
for i in rng.choice(len(newest_run["T"]), 8, replace=False):
    run = hrrr[hrrr["init_utc"] == pd.Timestamp(newest_run["init"][i]).tz_localize("UTC")].set_index("hour_utc")
    past, future = make_inputs(newest_run["T"][i], df["load_mw"], run, check_card)
    if not (np.allclose(past, P[i], atol=1e-5) and np.allclose(future, F[i], atol=1e-5)):
        raise ValueError("make_inputs does not match features at " + str(newest_run["T"][i]))
print("8 random samples: make_inputs == features | past", past.shape, "future", future.shape)

## Part 4 · The baseline to beat
**Same hour yesterday**, the leaderboard's yardstick.

In [ ]:
def same_hour_yesterday(d):
    target = d["T"].tz_convert(None).values[:, None] + (np.arange(1, H + 1) * np.timedelta64(1, "h"))[None, :]
    return df["load_mw"].reindex(pd.DatetimeIndex(target.ravel(), tz="UTC") - pd.Timedelta(hours=24)).values.reshape(-1, H)


valid = subset(newest_run, U("2025-01-01"), U("2026-01-01"))      # 2025: validation
test = subset(newest_run, U("2026-01-01"), U("2030-01-01"))       # 2026: the holdout, looked at once
for name, d in [("2025", valid), ("2026", test)]:
    print(f"same hour yesterday, average error {name}: {np.nanmean(np.abs(same_hour_yesterday(d) - d['y'])):.0f} MW")

## Part 5 · Build and fit the percentile network
One **member**: `Conv1D` (32 filters, 3 hours wide) + `MaxPooling1D` + `GRU(128)` read the past week into one summary;
`RepeatVector` hands that summary to each of the next 24 hours, next to that hour's weather, calendar and anchors; a
second `GRU(128)` walks the 24 hours; `TimeDistributed(Dense)` layers give 3 numbers per hour (10th, 50th, 90th). A
linear **skip** (`TimeDistributed(Dense)` straight from the future inputs, added at the end) lets the anchors pass
through when they're the best guess. Loss: the **pinball loss** (Part 2). The **ensemble** averages 5 members.

**The loss curve.** First, one member trained on 2023-2024 with early stopping on 2025, to see how training stops
(this is how the fixed 12 epochs were picked: the best 2025 loss came at epochs 9-19 across my runs).

In [ ]:
Q = np.array(QUANTILES, dtype="float32")


def pinball_loss(y_true, y_pred):
    error = keras.ops.expand_dims(y_true, -1) - y_pred
    return keras.ops.mean(keras.ops.maximum(Q * error, (Q - 1) * error))


def member(i, past_in, future_in):
    x = layers.Conv1D(SETTINGS["conv_filters"], 3, padding="same", activation="relu", name=f"m{i}_conv")(past_in)
    x = layers.MaxPooling1D(2, name=f"m{i}_pool")(x)
    summary = layers.GRU(SETTINGS["units"], name=f"m{i}_gru_past")(x)
    z = layers.Concatenate(name=f"m{i}_concat")([future_in, layers.RepeatVector(H, name=f"m{i}_repeat")(summary)])
    z = layers.GRU(SETTINGS["units"], return_sequences=True, name=f"m{i}_gru_future")(z)
    z = layers.TimeDistributed(layers.Dense(32, activation="relu"), name=f"m{i}_dense")(z)
    out = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name=f"m{i}_quantiles")(z)
    skip = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name=f"m{i}_skip")(future_in)
    return layers.Add(name=f"m{i}_add")([out, skip])


def training_arrays(train_end, card):
    parts = [features(subset(d, U("2000-01-01"), train_end), card) for d in (newest_run, previous_run)]
    return [np.concatenate([p[j] for p in parts]) for j in range(3)]


# one member, early stopping on 2025: the loss curve
card_2024 = make_card(U("2025-01-01"))
Xp, Xf, Y = training_arrays(U("2025-01-01"), card_2024)
Vp, Vf, VY = features(valid, card_2024)
keras.utils.set_random_seed(SETTINGS["seed"])
past_in = keras.Input((LB, Xp.shape[2]), name="past")
future_in = keras.Input((H, Xf.shape[2]), name="future")
one = keras.Model([past_in, future_in], member(0, past_in, future_in), name="one_member")
one.compile(optimizer="adam", loss=pinball_loss)
one.summary()
history = one.fit([Xp, Xf], Y, validation_data=([Vp, Vf], VY), epochs=30, batch_size=SETTINGS["batch_size"],
                  callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=2)
loss, val_loss = history.history["loss"], history.history["val_loss"]
plt.plot(range(1, len(loss) + 1), loss, "bo", label="Training loss (2023-2024)")
plt.plot(range(1, len(loss) + 1), val_loss, "b", label="Validation loss (2025)")
plt.axvline(SETTINGS["epochs"], color="gray", linestyle=":", label=f"fixed epochs for the ensemble ({SETTINGS['epochs']})")
plt.title("Figure 1. Loss curve: one member, pinball loss"); plt.xlabel("Epochs"); plt.ylabel("pinball loss (scaled)")
plt.legend(); plt.show()
print("best validation loss at epoch", int(np.argmin(val_loss)) + 1)

In [ ]:
def train_ensemble(train_end):
    """5 members, fixed epochs, on every issue hour whose 24 targets end before train_end; averaged in one keras.Model."""
    card = make_card(train_end)
    Xp, Xf, Y = training_arrays(train_end, card)
    past_in = keras.Input((LB, Xp.shape[2]), name="past")
    future_in = keras.Input((H, Xf.shape[2]), name="future")
    outs = []
    for i in range(SETTINGS["members"]):
        keras.utils.set_random_seed(SETTINGS["seed"] + i)
        out = member(i, past_in, future_in)
        net = keras.Model([past_in, future_in], out, name=f"net{i}")
        net.compile(optimizer="adam", loss=pinball_loss)
        start = time.time()
        net.fit([Xp, Xf], Y, epochs=SETTINGS["epochs"], batch_size=SETTINGS["batch_size"], verbose=0, shuffle=True)
        print(f"  member {i + 1}/{SETTINGS['members']}: {time.time() - start:.0f} s, {len(Y)} samples")
        outs.append(out)
    averaged = layers.Average(name="ensemble_average")(outs) if len(outs) > 1 else outs[0]
    return keras.Model([past_in, future_in], averaged, name="rama_llama_ensemble"), card


def load_ensemble(folder):
    """My trained ensemble + its card, from my repo."""
    os.makedirs(folder, exist_ok=True)
    for f in ["model.keras", "model_card.json"]:
        urllib.request.urlretrieve(REPO + "models/" + folder + "/" + f, folder + "/" + f)
    return keras.models.load_model(folder + "/model.keras", compile=False), json.load(open(folder + "/model_card.json"))


if RETRAIN:
    print("model_2024 (trained 2023-2024)"); model_2024, card_2024 = train_ensemble(U("2025-01-01"))
    print("model_2025 (trained 2023-2025)"); model_2025, card_2025 = train_ensemble(U("2026-01-01"))
    print("model_final (trained through Oct 2026)"); model_final, card_final = train_ensemble(U("2030-01-01"))
else:
    model_2024, card_2024 = load_ensemble("eval/trained_2023_2024")
    model_2025, card_2025 = load_ensemble("eval/trained_2023_2025")
    model_final, card_final = load_ensemble("live")
print("ensemble:", model_final.count_params(), "parameters |", len(model_final.layers), "layers")

## Part 6 · Calibrate the range on 2025
The rule from FP2: stretch the range around the expected value and keep the factor with the **lowest CRPS among those
with at least 80% of 2025's hours inside**.

In [ ]:
def forecast(model, card, d, widen=1.0, batch=1024):
    P, F, _ = features(d, card)
    mw = np.sort(model.predict([P, F], verbose=0, batch_size=batch) * card["load_std"] + card["load_mean"], -1)
    low, mid, high = mw[..., 0], mw[..., 1], mw[..., 2]
    return mid - widen * (mid - low), mid, mid + widen * (high - mid)


def score(name, d, low, mid, high):
    actual = d["y"]
    s = crps(actual, low, mid, high)
    return {"forecast": name, "avg miss (MW)": round(float(np.abs(actual - mid).mean()), 1),
            "inside (%), aim 80": round(100 * float(((actual >= low) & (actual <= high)).mean()), 1),
            "width (MW)": round(float((high - low).mean())), "CRPS (MW)": round(float(s.mean()), 1),
            "CRPS skill": round(crps_skill(s, np.abs(same_hour_yesterday(d) - actual)), 3)}


low_v, mid_v, high_v = forecast(model_2024, card_2024, valid)
factors = np.round(np.arange(0.9, 1.81, 0.05), 2)
curve = pd.DataFrame([score(f, valid, mid_v - f * (mid_v - low_v), mid_v, mid_v + f * (high_v - mid_v)) for f in factors])
ok = curve[curve["inside (%), aim 80"] >= 80]
best_widen = float(factors[ok["CRPS (MW)"].idxmin()]) if len(ok) else float(factors[-1])
plt.figure(figsize=(9, 4))
plt.plot(curve["inside (%), aim 80"], curve["CRPS (MW)"], "o-", markersize=3)
plt.scatter(curve.loc[factors == best_widen, "inside (%), aim 80"], curve.loc[factors == best_widen, "CRPS (MW)"], color="black", zorder=3, label=f"chosen: widen x{best_widen}")
plt.axvline(80, color="gray", linestyle=":"); plt.legend()
plt.xlabel("% of 2025 hours inside the range"); plt.ylabel("CRPS on 2025 (MW)"); plt.title("Figure 2. Calibrating the range on 2025")
plt.show()
yesterday_v = same_hour_yesterday(valid)
lo_c, hi_c = mid_v - best_widen * (mid_v - low_v), mid_v + best_widen * (high_v - mid_v)
scoreboard_2025 = pd.DataFrame([score("same hour yesterday", valid, yesterday_v, yesterday_v, yesterday_v),
                                score("no range (expected only)", valid, mid_v, mid_v, mid_v),
                                score("ensemble percentiles", valid, low_v, mid_v, high_v),
                                score(f"calibrated: widen x{best_widen}", valid, lo_c, mid_v, hi_c)]).set_index("forecast")
print("chosen range_widen:", best_widen)
scoreboard_2025

## Part 7 · One look at 2026, and a look inside the model
`model_2025` (trained 2023-2025, nothing re-tuned) on Jan to Oct 2026 with the HRRR runs as issued and the Part 6
widen. This is the honest test number. For reference, the course notebooks scored 0.64 (FP2 option B) and 0.68 (FP2b
champion) CRPS skill on 2026 with HRRR as issued.

In [ ]:
lo, mid, hi = forecast(model_2025, card_2025, test, best_widen)
actual_test = test["y"]
yesterday_t = same_hour_yesterday(test)
recent = subset(newest_run, U("2026-08-15"), U("2026-10-17"))
lr, mr, hr_ = forecast(model_2025, card_2025, recent, best_widen)
scoreboard_2026 = pd.DataFrame([score("same hour yesterday", test, yesterday_t, yesterday_t, yesterday_t),
                                score(f"ensemble, widen x{best_widen} (2026 Jan-Oct)", test, lo, mid, hi),
                                score("  same model, Aug 15 - Oct 15 2026 only", recent, lr, mr, hr_)]).set_index("forecast")
scoreboard_2026

In [ ]:
# Figure 3: actual vs expected, 1 hour and 24 hours ahead, colored by inside / outside the range
inside = (actual_test >= lo) & (actual_test <= hi)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, h in zip(ax, [0, 23]):
    a.scatter(mid[~inside[:, h], h], actual_test[~inside[:, h], h], s=2, alpha=0.4, color="tab:orange", label="outside the range")
    a.scatter(mid[inside[:, h], h], actual_test[inside[:, h], h], s=2, alpha=0.4, color="tab:blue", label="inside the range")
    lim = [actual_test.min(), actual_test.max()]
    a.plot(lim, lim, color="gray", linewidth=1)
    a.set_title(f"{h + 1} hour(s) ahead: {inside[:, h].mean():.0%} inside"); a.set_xlabel("expected (MW)"); a.set_ylabel("actual (MW)")
    a.legend(markerscale=6)
plt.suptitle("Figure 3. 2026 holdout: actual vs expected"); plt.tight_layout(); plt.show()

In [ ]:
# Figure 4: the range vs the actual load for one week: the forecast issued at 6 AM each day of the hottest week of 2026
local = test["T"].tz_convert("America/New_York")
hot = df.loc["2026-01-01":, "temp_f"].idxmax().tz_convert("America/New_York").normalize()
plt.figure(figsize=(12, 4))
for k in np.where((local.hour == 6) & (np.abs((local.normalize() - hot).days) <= 3))[0]:
    hrs = pd.date_range(test["T"][k] + pd.Timedelta(hours=1), periods=H, freq="h").tz_convert("America/New_York")
    plt.fill_between(hrs, lo[k], hi[k], color="tab:blue", alpha=0.3)
    plt.plot(hrs, mid[k], color="tab:blue", linewidth=1)
    plt.plot(hrs, actual_test[k], color="black", linewidth=1.5)
plt.title("Figure 4. Hottest week of 2026: blue = 80% range and expected value (issued 6 AM daily), black = actual")
plt.ylabel("MW"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
# Figure 5: permutation importance on 2026 (every 3rd forecast to keep it quick): shuffle one input column across
# forecasts and see how much worse the CRPS gets
sample = {k: v[::3] for k, v in test.items()}
P0, F0, _ = features(sample, card_2025)
y0 = sample["y"]


def crps_of(P, F):
    mw = np.sort(model_2025.predict([P, F], verbose=0, batch_size=2048) * card_2025["load_std"] + card_2025["load_mean"], -1)
    l, m, h = mw[..., 0], mw[..., 1], mw[..., 2]
    return float(crps(y0, m - best_widen * (m - l), m, m + best_widen * (h - m)).mean())


base_crps = crps_of(P0, F0)
order = np.random.default_rng(SETTINGS["seed"]).permutation(len(P0))
rows = []
for j, name in enumerate(FUTURE_COLUMNS):
    F1 = F0.copy(); F1[:, :, j] = F0[order, :, j]
    rows.append({"input": "future: " + name, "CRPS gets worse by (MW)": round(crps_of(P0, F1) - base_crps, 1)})
for j, name in enumerate(PAST_COLUMNS):
    P1 = P0.copy(); P1[:, :, j] = P0[order, :, j]
    rows.append({"input": "past week: " + name, "CRPS gets worse by (MW)": round(crps_of(P1, F0) - base_crps, 1)})
importance = pd.DataFrame(rows).sort_values("CRPS gets worse by (MW)")
plt.figure(figsize=(8, 7))
plt.barh(importance["input"], importance["CRPS gets worse by (MW)"], color="tab:blue")
plt.xlabel("how much worse the CRPS gets when this input is shuffled (MW)")
plt.title(f"Figure 5. Permutation importance on 2026 (CRPS {base_crps:.1f} MW with nothing shuffled)")
plt.tight_layout(); plt.show()

In [ ]:
# Table: the biggest misses of 2026 (worst hour of each day): under-forecasts (load above) and over-forecasts (below)
miss = actual_test - mid
order_of_misses = np.argsort(miss, axis=None)
rows = []
for kind, positions in [("under-forecast", order_of_misses[::-1]), ("over-forecast", order_of_misses)]:
    days = []
    for position in positions:
        k, h = np.unravel_index(position, miss.shape)
        target = test["T"][k] + pd.Timedelta(hours=int(h) + 1)
        day = target.tz_convert("America/New_York").date()
        if day in days:
            continue
        days.append(day)
        if len(days) > 5:
            break
        rows.append({"miss": kind, "hour (CT time)": target.tz_convert("America/New_York").strftime("%a %b %d %Y, %I %p"),
                     "hours ahead": int(h) + 1, "actual (MW)": round(float(actual_test[k, h])), "expected (MW)": round(float(mid[k, h])),
                     "low - high (MW)": f"{lo[k, h]:.0f} - {hi[k, h]:.0f}", "temp (F)": round(float(df.loc[target, "temp_f"]), 1)})
pd.DataFrame(rows)

## Part 8 · Save it for the cloud
The live model is `model_final`: the same recipe trained on every hour through Oct 2026 (the newest load shape), with
the Part 6 widen. `"weather": "hrrr"`: the cloud feeds it the newest HRRR run from the class file, like in training (if
that file is missing, the NWS forecast is used and sunlight is estimated from its cloud cover).

In [ ]:
card = dict(card_final)
card.update({"range_method": "lstm_quantiles", "quantiles": QUANTILES, "range_widen": best_widen, "weather": "hrrr",
             "members": SETTINGS["members"], "units": SETTINGS["units"], "epochs": SETTINGS["epochs"],
             "test_mae_mw": round(float(np.abs(actual_test - mid).mean()), 1),
             "test_2026": scoreboard_2026.reset_index().to_dict("records"),
             "trained_at": card_final.get("trained_at", pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"))})
model_final.save("model.keras")
with open("model_card.json", "w") as f:
    json.dump(card, f, indent=1)
reloaded = keras.models.load_model("model.keras", compile=False)
P, F, _ = features(subset(test, U("2026-10-01"), U("2026-10-17")), card)
print("reloaded model reproduces the predictions:", bool(np.allclose(reloaded.predict([P[:16], F[:16]], verbose=0),
                                                                       model_final.predict([P[:16], F[:16]], verbose=0), atol=1e-4)))
print("saved model.keras + model_card.json | range_method", card["range_method"], "| range_widen", card["range_widen"])

## Part 9 · Ship it to your energy project
**Once, before the first ship:** my fork's `forecast-24h-dl` has `make_inputs_rama_v1` (the copy of Part 3's
`make_inputs`); it redeploys automatically when that code changes on `main` (or: Actions > "Deploy forecast-24h-dl" >
Run workflow). **Deep learning only:** the forecast comes from `model.predict` on this Keras ensemble, run by
`forecast-24h-dl`.

In [ ]:
PROJECT_ID = "energy-raa20002-f26-515102"
UPLOAD = False                          # True to ship
BUCKET_NAME = PROJECT_ID
if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    bucket.blob("energy/models/model.keras").upload_from_filename("model.keras")
    bucket.blob("energy/models/model_card.json").upload_from_filename("model_card.json")
    stamp = card["trained_at"].replace(":", "-")
    bucket.blob("energy/models/history/model_" + stamp + ".keras").upload_from_filename("model.keras")
    bucket.blob("energy/models/history/model_card_" + stamp + ".json").upload_from_filename("model_card.json")
    print("Live model: gs://" + BUCKET_NAME + "/energy/models/model.keras | range_method:", card["range_method"])
    !gcloud scheduler jobs run forecast-24h-dl-hourly --location=us-central1 --project={PROJECT_ID}
else:
    print("UPLOAD is False: nothing shipped.")

In [ ]:
if UPLOAD:
    time.sleep(90)
    latest = pd.read_csv(io.BytesIO(bucket.blob("energy/forecasts/latest.csv").download_as_bytes()))
    print(latest[["target_hour_utc", "predicted_low_mw", "predicted_load_mw", "predicted_high_mw", "range_method"]].head(8))

## Your turn: what I tried (2025 validation, HRRR as issued; single networks unless noted)

| Network | 2025 CRPS (MW) | Note |
| :-- | --: | :-- |
| GRU encoder-decoder, 64 units | 64.6 / 61.9 | two random seeds: ~1.5 MW of noise between runs |
| same, LSTM instead of GRU | 67.5 | worse |
| same, without sunlight | 67.3 | sunlight helps ~3 MW |
| same, 128 units | 62.7 | small gain, kept |
| residual on yesterday / last week instead of the skip | 63.5 / 63.6 | a tie |
| holiday-aware inputs (past day-off flag, Black Friday) | 63.1 | a tie (post-Thanksgiving week 66.7 vs 67.3) |
| weather that actually happened (an oracle, not usable live) | 56.5 | what perfect weather would be worth: ~8 MW |
| **5 x GRU-128 averaged (the shipped recipe)** | **58.5** | averaging cut ~5 MW |